# Clase 3 — Demo: Pipeline completo con API real

**IADS 3 | Desarrollo de Sistemas de IA — Unidad 2**

Esta demo **sí necesita una API key real** (de OpenAI) y **consume créditos reales**, aunque el costo por esta prueba es mínimo (una sola llamada, modelo económico). Es la continuación directa de la Demo 1: ahí validamos datos escritos a mano; acá el LLM genera esos datos automáticamente a partir de un correo real de Ortelana, y Pydantic hace exactamente el mismo trabajo de guardia.

> **Antes de proyectar esto en clase:** correla una vez a solas para confirmar que tu API key funciona. Nunca pegues tu key directamente en una celda que vas a compartir en pantalla — usá el método de `getpass` de abajo, que la oculta mientras la escribís.

## Paso 1: instalar dependencias

In [ ]:
!pip install openai pydantic --quiet

## Paso 2: cargar la API key de forma segura

En Colab no usamos un archivo `.env` (eso es para cuando corrés el script en tu computadora). Acá usamos `getpass`, que te deja pegar la key sin que quede visible en pantalla ni guardada en el notebook.

In [ ]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Pegá tu OPENAI_API_KEY y presioná Enter: ")

## Paso 3: el contrato de datos (mismo `schemas.py` de siempre)

In [ ]:
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Literal, Optional


IntencionesPermitidas = Literal[
    "ALTA_DISTRIBUIDOR", "CONSULTA_DISTRIBUIDOR", "CONSULTA_STOCK", "RECLAMO_OPERATIVO"
]


class ConsultaClienteSchema(BaseModel):
    intencion_principal: IntencionesPermitidas = Field(
        description="Categoría raíz del mensaje según el árbol de intenciones del negocio."
    )
    identificador_fiscal: Optional[str] = Field(default=None, description="Identificador fiscal numérico (ej: CUIT de 11 dígitos).")
    razon_social_declarada: Optional[str] = Field(default=None, description="Nombre de la empresa o razón social provista por el remitente.")
    tipo_tela: Optional[str] = Field(default=None, description="Nombre comercial de la tela solicitada (ej: gabardina, lona, denim).")
    color: Optional[str] = Field(default=None, description="Color específico solicitado por el cliente.")
    confianza_clasificacion: float = Field(description="Nivel de certeza probabilística del modelo en la clasificación, de 0.0 a 1.0.")

    @field_validator("identificador_fiscal")
    @classmethod
    def validar_y_limpiar_idfiscal(cls, v):
        if v is None:
            return v
        idfiscal_limpio = "".join(filter(str.isdigit, v))
        if idfiscal_limpio and len(idfiscal_limpio) != 11:
            raise ValueError("El identificador fiscal debe contener exactamente 11 dígitos numéricos.")
        return idfiscal_limpio

## Paso 4: el correo real (desestructurado) de un distribuidor

In [ ]:
correo_recibido = """
Hola gente de Ortelana!! Cómo andan??
Soy del taller 'Modas del Sur', mi CUIT es el 20-34556789-2.
Queríamos ver si tienen stock de denim de 12 en azul índigo.
Necesitamos cotizar unos 15 rollos urgente porque se nos cae un cliente grande.
Calculamos una compra de medio millón si nos hacen buen precio.
Avisen porfa!! Gracias!!
"""
print(correo_recibido)

## Paso 5: llamada real a la API con Structured Outputs

`response_format=ConsultaClienteSchema` es la "Garantía del Proveedor": OpenAI restringe matemáticamente los tokens que puede generar el modelo para que respeten el esquema. Lo que sale de `.parsed` ya es un objeto Pydantic — no un string que hay que parsear a mano.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

try:
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Sos un agente de clasificación semántica para Ortelana Textil. "
                    "Tu único objetivo es extraer datos estructurados conforme al esquema proporcionado."
                ),
            },
            {"role": "user", "content": correo_recibido},
        ],
        response_format=ConsultaClienteSchema,
    )

    datos: ConsultaClienteSchema = completion.choices[0].message.parsed

    print("--- PIPELINE EXITOSO ---")
    print(f"Intención:            {datos.intencion_principal}")
    print(f"Identificador fiscal: {datos.identificador_fiscal}")
    print(f"Razón social:         {datos.razon_social_declarada}")
    print(f"Tipo de tela:         {datos.tipo_tela}")
    print(f"Color:                {datos.color}")
    print(f"Confianza:            {datos.confianza_clasificacion:.2f}")

except ValidationError as e:
    print(f"ERROR: Los datos violaron el schema de Ortelana. Detalles: {e}")
except Exception as e:
    print(f"Falla de comunicación con el proveedor: {e}")

### Para probar en vivo con el grupo

Volvé al Paso 4 y cambiá el texto de `correo_recibido` por alguno de los 15 correos "rotos" que ya vimos en la Clase 2 (el hostil, el que pide cancelar cuenta, el que mezcla varios pedidos). Volvé a correr desde ahí — la diferencia con la Clase 2 es que ahora, aunque el LLM devuelva cualquier cosa, **Pydantic no deja pasar un dato mal formado hacia el resultado final**.